# Labelled data viewer — SimuScanAgent

Inspect the ground truth of a synthetic AFM dataset: the original image and the same
image with its labels drawn on top, side by side. A dataset is annotated in several
formats at once, so the label source is selectable — COCO polygons, the two YOLO
exports, the instance map and the semantic UNet mask all read back into one common
structure and render the same way, which makes it easy to check that a conversion
did not lose or shift anything.

**Author:** Ruben Millan-Solsona

Standalone notebook: it imports nothing from `src/`, only `numpy`, `matplotlib` and
`Pillow`, plus `PyYAML`, `pandas` and `ipywidgets` when they are available.

| source | path inside the dataset | what it gives |
|---|---|---|
| `coco` | `annotations/instances_<split>.json` | polygons + boxes + class + per-instance metadata |
| `yolo_seg` | `_yolo_seg/labels/<split>/*.txt` | normalised polygons + class |
| `yolo_det` | `_yolo_det/labels/<split>/*.txt` | boxes only, normalised + class |
| `instances` | `instances/<split>/*.png` (uint16) | **instance map**: one id per object/grain |
| `unet` | `_unet/masks/<split>/*.png` (uint8) | **semantic** mask: one value per class |

Toggles for box, filled mask, segmentation outline and label text; colour by instance
or by class; adjustable opacity and class filter. Further down there is an interactive
panel and a `label -> count` summary.

> The interactive panel uses `ipywidgets`, already declared in `pyproject.toml` (`uv sync`).
> Without it everything still works by editing the configuration cell.

In [ ]:
# ─────────────────────────────── CONFIGURATION ───────────────────────────────
# Edit this and re-run the "show" cell (or use the interactive panel below).

DATASETS_ROOT = None      # None = autodetect <repo>/data/Datasets ; or an absolute path
DATASET       = None      # None = the most recent one ; or the folder name
SPLIT         = "train"   # "train" | "val" | "test"
SOURCE        = "coco"    # "coco" | "yolo_seg" | "yolo_det" | "instances" | "unet"
IMAGE         = 0         # index (int) or file name ("000000.png")

# ── what gets drawn ──
SHOW_BOX      = True      # bounding box
SHOW_MASK     = True      # filled mask / polygon
SHOW_CONTOUR  = True      # segmentation outline
SHOW_LABEL    = True      # label text
SHOW_INST_ID  = False     # append the instance id to the text

COLOR_BY      = "instance"   # "instance" (one colour per object) | "class" (one per class)
MASK_ALPHA    = 0.45         # fill opacity
CONTOUR_WIDTH = 1            # outline thickness, in px
FILTER_LABELS = None         # None = every class ; or a list ["square", "triangle"]

ORIGINAL_SIDE = "right"      # side the original image goes on: "right" | "left"
FIGSIZE       = (15.0, 7.6)
MAX_LABELS    = 60           # above this the text is skipped (unreadable and slow)
MAX_BOXES     = 3000         # above this the boxes are skipped

In [ ]:
from __future__ import annotations

import colorsys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw

try:
    import yaml                      # only to read the YOLO dataset.yaml files
except ImportError:
    yaml = None
try:
    import pandas as pd              # only so the summary renders as a table
except ImportError:
    pd = None

plt.rcParams["figure.facecolor"] = "white"
IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")


# ───────────────────────────── colour and geometry ─────────────────────────────

_GOLDEN = 0.61803398875


def instance_color(i: int):
    """Well separated colours even with hundreds of touching instances."""
    h = (i * _GOLDEN) % 1.0
    s = 0.62 + 0.28 * (((i * 7) % 3) / 2.0)
    v = 0.78 + 0.22 * ((i * 5) % 2)
    return colorsys.hsv_to_rgb(h, s, v)


_CLASS_CMAP = plt.get_cmap("tab10")


def class_color(k: int):
    return tuple(_CLASS_CMAP(k % 10)[:3])


def boundary_mask(id_map: np.ndarray, width: int = 1) -> np.ndarray:
    """Border pixels of every region in `id_map` (0 = background). No scipy needed."""
    d = np.zeros(id_map.shape, bool)
    d[:-1, :] |= id_map[:-1, :] != id_map[1:, :]
    d[1:, :] |= id_map[1:, :] != id_map[:-1, :]
    d[:, :-1] |= id_map[:, :-1] != id_map[:, 1:]
    d[:, 1:] |= id_map[:, 1:] != id_map[:, :-1]
    b = d & (id_map > 0)
    # objects cut by the frame get an outline too
    b[0, :] |= id_map[0, :] > 0
    b[-1, :] |= id_map[-1, :] > 0
    b[:, 0] |= id_map[:, 0] > 0
    b[:, -1] |= id_map[:, -1] > 0
    for _ in range(max(0, int(width) - 1)):
        g = b.copy()
        g[:-1, :] |= b[1:, :]
        g[1:, :] |= b[:-1, :]
        g[:, :-1] |= b[:, 1:]
        g[:, 1:] |= b[:, :-1]
        b = g & (id_map > 0)
    return b


def bboxes_from_idmap(id_map: np.ndarray, n: int) -> dict:
    """Box (x, y, w, h) for every id 1..n-1, in a single pass."""
    fg = id_map > 0
    if not fg.any():
        return {}
    ys, xs = np.nonzero(fg)
    ids = id_map[ys, xs].astype(np.int64)
    big = np.iinfo(np.int32).max
    x0 = np.full(n, big, np.int64)
    y0 = np.full(n, big, np.int64)
    x1 = np.full(n, -1, np.int64)
    y1 = np.full(n, -1, np.int64)
    np.minimum.at(x0, ids, xs)
    np.minimum.at(y0, ids, ys)
    np.maximum.at(x1, ids, xs)
    np.maximum.at(y1, ids, ys)
    out = {}
    for i in range(1, n):
        if x1[i] >= 0:
            out[i] = (float(x0[i]), float(y0[i]),
                      float(x1[i] - x0[i] + 1), float(y1[i] - y0[i] + 1))
    return out

In [ ]:
# ───────────────────────────── dataset discovery ─────────────────────────────

SOURCE_NAMES = ["coco", "yolo_seg", "yolo_det", "instances", "unet"]


def find_datasets_root(explicit=None) -> Path:
    if explicit:
        return Path(explicit).expanduser().resolve()
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        cand = base / "data" / "Datasets"
        if cand.is_dir():
            return cand
    raise FileNotFoundError(
        "Cannot find data/Datasets walking up from the current directory. "
        "Set DATASETS_ROOT to an absolute path."
    )


def list_datasets(root: Path) -> list:
    """Folders that look like a dataset (they have images/ or annotations/)."""
    return [d.name for d in sorted(root.iterdir())
            if d.is_dir() and ((d / "images").is_dir() or (d / "annotations").is_dir())]


def source_path(ds: Path, source: str, split: str) -> Path:
    return {
        "coco":      ds / "annotations" / f"instances_{split}.json",
        "yolo_seg":  ds / "_yolo_seg" / "labels" / split,
        "yolo_det":  ds / "_yolo_det" / "labels" / split,
        "instances": ds / "instances" / split,
        "unet":      ds / "_unet" / "masks" / split,
    }[source]


def available_sources(ds: Path, split: str) -> list:
    return [s for s in SOURCE_NAMES if source_path(ds, s, split).exists()]


def available_splits(ds: Path) -> list:
    splits = [s for s in ("train", "val", "test") if (ds / "images" / s).is_dir()]
    return splits or ["train"]


def images_dir(ds: Path, split: str, source: str = "coco") -> Path:
    """The base image always comes from images/<split> when it exists, so the
    left/right comparison shows the same image for every source."""
    base = ds / "images" / split
    if base.is_dir():
        return base
    alt = {
        "yolo_seg": ds / "_yolo_seg" / "images" / split,
        "yolo_det": ds / "_yolo_det" / "images" / split,
        "unet":     ds / "_unet" / "images" / split,
    }.get(source)
    if alt and alt.is_dir():
        return alt
    raise FileNotFoundError(f"No image folder for {ds.name}/{split}")


def list_images(ds: Path, split: str, source: str = "coco") -> list:
    d = images_dir(ds, split, source)
    return sorted(p.name for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)


def dataset_class_names(ds: Path, source: str = "coco") -> list:
    """Class names in index order (a YOLO class index is a position in this list)."""
    yml = {"yolo_seg": ds / "_yolo_seg" / "dataset.yaml",
           "yolo_det": ds / "_yolo_det" / "dataset.yaml"}.get(source)
    if yml is not None and yml.is_file() and yaml is not None:
        names = yaml.safe_load(yml.read_text(encoding="utf-8")).get("names")
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names:
            return list(names)
    info = ds / "dataset_info.json"
    if info.is_file():
        names = json.loads(info.read_text(encoding="utf-8")).get("class_names")
        if names:
            return list(names)
    for sp in ("train", "val", "test"):
        js = ds / "annotations" / f"instances_{sp}.json"
        if js.is_file():
            cats = sorted(json.loads(js.read_text(encoding="utf-8"))["categories"],
                          key=lambda c: c["id"])
            return [c["name"] for c in cats]
    return []


def resolve_dataset(root: Path, name=None) -> Path:
    names = list_datasets(root)
    if not names:
        raise FileNotFoundError(f"No datasets under {root}")
    if name is None:
        return max((root / n for n in names), key=lambda p: p.stat().st_mtime)
    return root / name


def load_image(ds: Path, split: str, name: str, source: str = "coco") -> np.ndarray:
    d = images_dir(ds, split, source)
    p = d / name
    if not p.is_file():                  # in case the extension does not match
        stem = Path(name).stem
        cands = [q for q in d.iterdir() if q.stem == stem and q.suffix.lower() in IMG_EXTS]
        if not cands:
            raise FileNotFoundError(p)
        p = cands[0]
    arr = np.array(Image.open(p))
    if arr.ndim == 3:
        arr = arr[..., :3]
    return arr

In [ ]:
# ───────────────────── readers: every source → the same structure ─────────────────────
#
# A "record" is a dict with:
#   label       class name
#   class_id    class index (None when the source does not give one)
#   instance_id id of the instance within the image
#   bbox        (x, y, w, h) in px, or None
#   polygons    list of (N, 2) arrays in px, or None
#   extra       loose metadata (COCO: grain_id, eq_diameter_px, mean_z, ...)

_COCO_CACHE = {}

_EXTRA_KEYS = ("grain_id", "eq_diameter_px", "eccentricity", "flatness", "mean_z",
               "roughness_rms_actual", "n_sides", "instance_index", "area",
               "radius_tip", "theta", "is_double")


def _coco_index(ds: Path, split: str):
    key = (str(ds), split)
    if key not in _COCO_CACHE:
        js = source_path(ds, "coco", split)
        data = json.loads(js.read_text(encoding="utf-8"))
        cats = {c["id"]: c["name"] for c in data["categories"]}
        order = {c["id"]: i for i, c in enumerate(sorted(data["categories"], key=lambda c: c["id"]))}
        by_img = {}
        id2name = {im["id"]: im["file_name"] for im in data["images"]}
        for ann in data["annotations"]:
            by_img.setdefault(id2name[ann["image_id"]], []).append(ann)
        _COCO_CACHE[key] = {"cats": cats, "order": order, "by_img": by_img,
                            "images": [im["file_name"] for im in data["images"]]}
    return _COCO_CACHE[key]


def _poly_from_flat(flat):
    a = np.asarray(flat, float)
    if a.size < 6:
        return None
    return a.reshape(-1, 2)


def records_coco(ds: Path, split: str, name: str, shape) -> list:
    idx = _coco_index(ds, split)
    out = []
    for ann in idx["by_img"].get(name, []):
        polys = []
        seg = ann.get("segmentation")
        if isinstance(seg, list):
            for s in seg:
                p = _poly_from_flat(s)
                if p is not None:
                    polys.append(p)
        out.append({
            "label": ann.get("category_name") or idx["cats"].get(ann["category_id"], str(ann["category_id"])),
            "class_id": idx["order"].get(ann["category_id"]),
            "instance_id": int(ann.get("grain_id", ann["id"])),
            "bbox": tuple(float(v) for v in ann["bbox"]) if ann.get("bbox") else None,
            "polygons": polys or None,
            "extra": {k: ann[k] for k in _EXTRA_KEYS if k in ann},
        })
    return out


def _yolo_lines(ds: Path, split: str, name: str, source: str):
    f = source_path(ds, source, split) / (Path(name).stem + ".txt")
    if not f.is_file():
        return []
    return [ln.split() for ln in f.read_text(encoding="utf-8").splitlines() if ln.strip()]


def records_yolo_seg(ds: Path, split: str, name: str, shape) -> list:
    H, W = shape[:2]
    names = dataset_class_names(ds, "yolo_seg")
    out = []
    for i, parts in enumerate(_yolo_lines(ds, split, name, "yolo_seg")):
        cls = int(float(parts[0]))
        coords = np.asarray(parts[1:], float)
        if coords.size < 6:
            continue
        poly = coords.reshape(-1, 2) * np.array([W, H])
        x0, y0 = poly.min(0)
        x1, y1 = poly.max(0)
        out.append({
            "label": names[cls] if cls < len(names) else f"class_{cls}",
            "class_id": cls,
            "instance_id": i,
            "bbox": (float(x0), float(y0), float(x1 - x0), float(y1 - y0)),
            "polygons": [poly],
            "extra": {},
        })
    return out


def records_yolo_det(ds: Path, split: str, name: str, shape) -> list:
    H, W = shape[:2]
    names = dataset_class_names(ds, "yolo_det")
    out = []
    for i, parts in enumerate(_yolo_lines(ds, split, name, "yolo_det")):
        cls = int(float(parts[0]))
        xc, yc, w, h = (float(v) for v in parts[1:5])
        out.append({
            "label": names[cls] if cls < len(names) else f"class_{cls}",
            "class_id": cls,
            "instance_id": i,
            "bbox": ((xc - w / 2) * W, (yc - h / 2) * H, w * W, h * H),
            "polygons": None,
            "extra": {},
        })
    return out


def records_instances(ds: Path, split: str, name: str, shape):
    """uint16 instance map. Labels come from the COCO file via `instance_index`."""
    p = source_path(ds, "instances", split) / (Path(name).stem + ".png")
    raw = np.array(Image.open(p))
    ids = np.unique(raw)
    ids = ids[ids > 0]

    by_index = {}
    js = source_path(ds, "coco", split)
    if js.is_file():
        for ann in _coco_index(ds, split)["by_img"].get(name, []):
            if "instance_index" in ann:
                by_index[int(ann["instance_index"])] = ann

    names = dataset_class_names(ds, "coco")
    cats = _coco_index(ds, split)["cats"] if js.is_file() else {}
    order = _coco_index(ds, split)["order"] if js.is_file() else {}

    id_map = np.zeros(raw.shape, np.int32)
    records = []
    for k, v in enumerate(ids):
        id_map[raw == v] = k + 1
        ann = by_index.get(int(v))
        if ann is not None:
            label = ann.get("category_name") or cats.get(ann["category_id"], "obj")
            cid = order.get(ann["category_id"])
            extra = {kk: ann[kk] for kk in _EXTRA_KEYS if kk in ann}
        else:
            label, cid, extra = (names[0] if len(names) == 1 else "instance"), None, {}
        records.append({"label": label, "class_id": cid, "instance_id": int(v),
                        "bbox": None, "polygons": None, "extra": extra})
    return records, id_map


def records_unet(ds: Path, split: str, name: str, shape):
    """uint8 semantic mask: 0 = background, v = class v-1. One region per class."""
    p = source_path(ds, "unet", split) / (Path(name).stem + ".png")
    raw = np.array(Image.open(p))
    names = dataset_class_names(ds, "coco")
    vals = np.unique(raw)
    vals = vals[vals > 0]
    id_map = np.zeros(raw.shape, np.int32)
    records = []
    for k, v in enumerate(vals):
        id_map[raw == v] = k + 1
        cid = int(v) - 1
        records.append({
            "label": names[cid] if cid < len(names) else f"class_{cid}",
            "class_id": cid, "instance_id": int(v),
            "bbox": None, "polygons": None, "extra": {"semantic": True},
        })
    return records, id_map


def rasterize(records: list, shape) -> np.ndarray:
    """Paint polygons (or boxes, when there is no polygon) into an id map: record k → k+1."""
    im = Image.new("I", (int(shape[1]), int(shape[0])), 0)
    d = ImageDraw.Draw(im)
    for k, r in enumerate(records):
        polys = r.get("polygons")
        if polys:
            for poly in polys:
                if len(poly) >= 3:
                    d.polygon([(float(x), float(y)) for x, y in poly], fill=k + 1)
        elif r.get("bbox"):
            x, y, w, h = r["bbox"]
            d.rectangle([float(x), float(y), float(x + w), float(y + h)], fill=k + 1)
    return np.array(im, dtype=np.int32)


def build_scene(ds: Path, split: str, name: str, source: str, filter_labels=None):
    """→ (image, records, id_map) with id_map[k+1] == records[k]."""
    img = load_image(ds, split, name, source)
    shape = img.shape[:2]

    if source == "instances":
        records, id_map = records_instances(ds, split, name, shape)
    elif source == "unet":
        records, id_map = records_unet(ds, split, name, shape)
    else:
        records = {"coco": records_coco, "yolo_seg": records_yolo_seg,
                   "yolo_det": records_yolo_det}[source](ds, split, name, shape)
        id_map = rasterize(records, shape)

    if filter_labels:
        keep = set(filter_labels)
        sel = [k for k, r in enumerate(records) if r["label"] in keep]
        remap = np.zeros(len(records) + 1, np.int32)
        for new, old in enumerate(sel):
            remap[old + 1] = new + 1
        id_map = remap[id_map]
        records = [records[k] for k in sel]

    # boxes the source did not provide (grey maps), and any that are missing
    missing = [k for k, r in enumerate(records) if r["bbox"] is None]
    if missing:
        boxes = bboxes_from_idmap(id_map, len(records) + 1)
        for k in missing:
            records[k]["bbox"] = boxes.get(k + 1)

    areas = np.bincount(id_map.ravel(), minlength=len(records) + 1)
    for k, r in enumerate(records):
        r["area_px"] = int(areas[k + 1])
    return img, records, id_map

In [ ]:
# ───────────────────────────── label → count summary ─────────────────────────────

def summarize(records: list):
    """Count per label, with mean and median area in px."""
    rows = {}
    for r in records:
        acc = rows.setdefault(r["label"], {"label": r["label"], "count": 0, "areas": []})
        acc["count"] += 1
        acc["areas"].append(r.get("area_px", 0))
    out = []
    for acc in rows.values():
        a = np.asarray(acc["areas"], float)
        out.append({"label": acc["label"], "count": acc["count"],
                    "mean_area_px": round(float(a.mean()), 1) if a.size else 0.0,
                    "median_area_px": round(float(np.median(a)), 1) if a.size else 0.0,
                    "% of frame": None})
    out.sort(key=lambda d: -d["count"])
    return out


def show_summary(records: list, shape=None, title: str = ""):
    rows = summarize(records)
    if shape is not None:
        total = float(shape[0] * shape[1])
        for r in rows:
            r["% of frame"] = round(100.0 * r["count"] * r["mean_area_px"] / total, 2)
    else:
        for r in rows:
            r.pop("% of frame", None)
    total_n = sum(r["count"] for r in rows)
    if title:
        print(title)
    print(f"objects: {total_n}   classes: {len(rows)}")
    if pd is not None:
        try:
            from IPython.display import display
            display(pd.DataFrame(rows))
            return rows
        except ImportError:
            pass
    cols = list(rows[0].keys()) if rows else []
    if cols:
        w = {c: max(len(str(c)), *(len(str(r[c])) for r in rows)) for c in cols}
        print("  ".join(str(c).ljust(w[c]) for c in cols))
        for r in rows:
            print("  ".join(str(r[c]).ljust(w[c]) for c in cols))
    return rows


def summarize_split(ds: Path, split: str, source: str, limit=None, verbose=True):
    """Summary over a whole split (walks the images; grey maps can be slow)."""
    names = list_images(ds, split, source)
    if limit:
        names = names[:limit]
    counts, areas, n_img = {}, {}, 0
    for i, name in enumerate(names):
        try:
            _, records, _ = build_scene(ds, split, name, source)
        except FileNotFoundError:
            continue
        n_img += 1
        for r in records:
            counts[r["label"]] = counts.get(r["label"], 0) + 1
            areas.setdefault(r["label"], []).append(r.get("area_px", 0))
        if verbose and (i + 1) % 25 == 0:
            print(f"  ... {i + 1}/{len(names)}")
    rows = [{"label": k, "count": v,
             "per image": round(v / max(n_img, 1), 2),
             "mean_area_px": round(float(np.mean(areas[k])), 1)}
            for k, v in sorted(counts.items(), key=lambda kv: -kv[1])]
    print(f"{ds.name} / {split} / {source}: {n_img} images, "
          f"{sum(counts.values())} objects")
    if pd is not None:
        try:
            from IPython.display import display
            display(pd.DataFrame(rows))
            return rows
        except ImportError:
            pass
    for r in rows:
        print(r)
    return rows

In [ ]:
# ───────────────────────────── drawing the overlay ─────────────────────────────

def record_colors(records: list, color_by: str = "instance") -> np.ndarray:
    cols = np.zeros((len(records) + 1, 3), float)
    for k, r in enumerate(records):
        if color_by == "class" and r.get("class_id") is not None:
            cols[k + 1] = class_color(r["class_id"])
        else:
            cols[k + 1] = instance_color(r.get("instance_id", k))
    return cols


def as_rgb(img: np.ndarray) -> np.ndarray:
    a = img.astype(float)
    if a.max() > 1.5:
        a = a / 255.0
    if a.ndim == 2:
        a = np.dstack([a] * 3)
    return np.clip(a, 0, 1)


def overlay_image(img, records, id_map, colors, *, show_mask=True, show_contour=True,
                  mask_alpha=0.45, contour_width=1) -> np.ndarray:
    base = as_rgb(img)
    out = base.copy()
    fg = id_map > 0
    if show_mask and fg.any():
        out[fg] = base[fg] * (1.0 - mask_alpha) + colors[id_map[fg]] * mask_alpha
    if show_contour and fg.any():
        b = boundary_mask(id_map, contour_width)
        out[b] = colors[id_map[b]]
    return np.clip(out, 0, 1)


def draw_annotations(ax, records, colors, *, show_box=True, show_label=True,
                     show_inst_id=False, max_labels=60, max_boxes=3000):
    if show_box and records:
        if len(records) > max_boxes:
            print(f"[note] {len(records)} objects > MAX_BOXES={max_boxes}: skipping the boxes.")
        else:
            patches, edge = [], []
            for k, r in enumerate(records):
                if not r.get("bbox"):
                    continue
                x, y, w, h = r["bbox"]
                patches.append(Rectangle((x - 0.5, y - 0.5), w, h))
                edge.append(colors[k + 1])
            if patches:
                ax.add_collection(PatchCollection(patches, facecolor="none",
                                                  edgecolors=edge, linewidths=1.2))
    if show_label and records:
        if len(records) > max_labels:
            print(f"[note] {len(records)} objects > MAX_LABELS={max_labels}: skipping the text.")
            return
        for k, r in enumerate(records):
            if not r.get("bbox"):
                continue
            x, y, _, _ = r["bbox"]
            txt = r["label"]
            if show_inst_id:
                txt = f"{txt} #{r.get('instance_id', k)}"
            ax.text(x, y - 2, txt, fontsize=8, color="black", va="bottom", ha="left",
                    bbox=dict(boxstyle="square,pad=0.15", facecolor=colors[k + 1],
                              edgecolor="none", alpha=0.85))


def show(dataset=None, split=None, source=None, image=None, *, datasets_root=None,
         show_box=None, show_mask=None, show_contour=None, show_label=None,
         show_inst_id=None, color_by=None, mask_alpha=None, contour_width=None,
         filter_labels=None, original_side=None, figsize=None,
         max_labels=None, max_boxes=None, summary=True, return_fig=False):
    """Original image and overlay side by side, plus the object summary."""
    root = find_datasets_root(datasets_root if datasets_root is not None else DATASETS_ROOT)
    ds = resolve_dataset(root, dataset if dataset is not None else DATASET)
    split = split or SPLIT
    source = source or SOURCE
    if source not in available_sources(ds, split):
        raise FileNotFoundError(
            f"{ds.name}/{split} has no '{source}' source. "
            f"Available: {available_sources(ds, split)}"
        )

    names = list_images(ds, split, source)
    image = IMAGE if image is None else image
    name = names[int(image) % len(names)] if isinstance(image, (int, np.integer)) else str(image)

    flt = FILTER_LABELS if filter_labels is None else filter_labels
    img, records, id_map = build_scene(ds, split, name, source, flt)
    colors = record_colors(records, COLOR_BY if color_by is None else color_by)

    over = overlay_image(
        img, records, id_map, colors,
        show_mask=SHOW_MASK if show_mask is None else show_mask,
        show_contour=SHOW_CONTOUR if show_contour is None else show_contour,
        mask_alpha=MASK_ALPHA if mask_alpha is None else mask_alpha,
        contour_width=CONTOUR_WIDTH if contour_width is None else contour_width,
    )

    side = original_side or ORIGINAL_SIDE
    fig, axes = plt.subplots(1, 2, figsize=figsize or FIGSIZE, constrained_layout=True)
    ax_over, ax_orig = (axes[0], axes[1]) if side == "right" else (axes[1], axes[0])

    ax_orig.imshow(as_rgb(img), interpolation="nearest")
    ax_orig.set_title(f"original — {name}")
    ax_over.imshow(over, interpolation="nearest")
    ax_over.set_title(f"labels ({source}) — {len(records)} objects")

    draw_annotations(
        ax_over, records, colors,
        show_box=SHOW_BOX if show_box is None else show_box,
        show_label=SHOW_LABEL if show_label is None else show_label,
        show_inst_id=SHOW_INST_ID if show_inst_id is None else show_inst_id,
        max_labels=MAX_LABELS if max_labels is None else max_labels,
        max_boxes=MAX_BOXES if max_boxes is None else max_boxes,
    )
    for ax in axes:
        ax.set_xlim(-0.5, img.shape[1] - 0.5)
        ax.set_ylim(img.shape[0] - 0.5, -0.5)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"{ds.name}  ·  {split}  ·  {source}  ·  {name}", fontsize=11)
    plt.show()

    if summary:
        show_summary(records, img.shape[:2])
    return (fig, records) if return_fig else records

In [ ]:
# ─────────────── inventory: which datasets exist and what labels each one carries ───────────────

ROOT = find_datasets_root(DATASETS_ROOT)
print(f"root: {ROOT}\n")
for n in list_datasets(ROOT):
    d = ROOT / n
    splits = available_splits(d)
    srcs = sorted({s for sp in splits for s in available_sources(d, sp)},
                  key=SOURCE_NAMES.index)
    try:
        n_img = len(list_images(d, splits[0], srcs[0] if srcs else "coco"))
    except Exception:
        n_img = 0
    print(f"{n:<28} splits={','.join(splits):<16} sources={','.join(srcs) or '-':<40} "
          f"classes={dataset_class_names(d)} ({n_img} img in {splits[0]})")

In [ ]:
# ─────────────────────────── show (uses the CONFIGURATION above) ───────────────────────────
# Any parameter can be overridden here without touching the config cell, e.g.:
#   show(source="instances", image=3, show_label=False, color_by="class")

records = show()

## Interactive panel

Uses `ipywidgets`, already a project dependency. If the kernel cannot find it:

```
uv sync --native-tls
```

then restart the kernel. Without `ipywidgets` the cell only prints a note and
everything else still works through `show(...)`.

The widgets are kept in `panel`, so they can also be driven from code:
`panel["image"].value = "000007.png"`, `panel["source"].value = "instances"`.

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False
    print("ipywidgets is not installed -> panel disabled.\n"
          "  uv sync --native-tls          (then restart the kernel)\n"
          "Meanwhile use show(source=..., image=..., show_box=False, ...).")


def build_panel():
    root = find_datasets_root(DATASETS_ROOT)
    ds_names = list_datasets(root)
    start = resolve_dataset(root, DATASET).name

    w_ds = W.Dropdown(options=ds_names, value=start, description="dataset",
                      layout=W.Layout(width="330px"))
    w_split = W.Dropdown(options=["train"], description="split", layout=W.Layout(width="180px"))
    w_src = W.Dropdown(options=["coco"], description="source", layout=W.Layout(width="200px"))
    w_img = W.Dropdown(options=["-"], description="image", layout=W.Layout(width="230px"))
    b_prev = W.Button(description="◀", layout=W.Layout(width="42px"))
    b_next = W.Button(description="▶", layout=W.Layout(width="42px"))

    w_box = W.Checkbox(value=SHOW_BOX, description="box", indent=False,
                       layout=W.Layout(width="110px"))
    w_mask = W.Checkbox(value=SHOW_MASK, description="mask", indent=False,
                        layout=W.Layout(width="110px"))
    w_cont = W.Checkbox(value=SHOW_CONTOUR, description="outline", indent=False,
                        layout=W.Layout(width="110px"))
    w_lab = W.Checkbox(value=SHOW_LABEL, description="label", indent=False,
                       layout=W.Layout(width="110px"))
    w_iid = W.Checkbox(value=SHOW_INST_ID, description="inst. id", indent=False,
                       layout=W.Layout(width="110px"))

    w_color = W.ToggleButtons(options=[("by instance", "instance"), ("by class", "class")],
                              value=COLOR_BY, description="colour")
    w_alpha = W.FloatSlider(value=MASK_ALPHA, min=0.0, max=1.0, step=0.05,
                            description="alpha", continuous_update=False,
                            layout=W.Layout(width="280px"))
    w_width = W.IntSlider(value=CONTOUR_WIDTH, min=1, max=5, description="outline px",
                          continuous_update=False, layout=W.Layout(width="230px"))
    w_side = W.ToggleButtons(options=[("original right", "right"), ("original left", "left")],
                             value=ORIGINAL_SIDE, description="side")
    w_filter = W.SelectMultiple(options=[], description="classes",
                                layout=W.Layout(width="230px", height="86px"))
    w_sum = W.Checkbox(value=True, description="summary", indent=False,
                       layout=W.Layout(width="110px"))
    out = W.Output()
    _busy = {"v": False}

    def ds_path():
        return root / w_ds.value

    def sync_split(*_):
        _busy["v"] = True
        try:
            opts = available_splits(ds_path())
            w_split.options = opts
            if w_split.value not in opts:
                w_split.value = opts[0]
        finally:
            _busy["v"] = False
        sync_source()

    def sync_source(*_):
        _busy["v"] = True
        try:
            opts = available_sources(ds_path(), w_split.value) or ["coco"]
            w_src.options = opts
            if w_src.value not in opts:
                w_src.value = opts[0]
        finally:
            _busy["v"] = False
        sync_images()

    def sync_images(*_):
        _busy["v"] = True
        try:
            names = list_images(ds_path(), w_split.value, w_src.value)
            w_img.options = names or ["-"]
            if w_img.value not in w_img.options:
                w_img.value = w_img.options[0]
            cls = dataset_class_names(ds_path(), w_src.value)
            w_filter.options = cls
            w_filter.value = ()
        finally:
            _busy["v"] = False
        refresh()

    def step(delta):
        def _cb(_):
            opts = list(w_img.options)
            if len(opts) > 1:
                w_img.value = opts[(opts.index(w_img.value) + delta) % len(opts)]
        return _cb

    def refresh(*_):
        if _busy["v"]:
            return
        with out:
            clear_output(wait=True)
            try:
                show(dataset=w_ds.value, split=w_split.value, source=w_src.value,
                     image=w_img.value, show_box=w_box.value, show_mask=w_mask.value,
                     show_contour=w_cont.value, show_label=w_lab.value,
                     show_inst_id=w_iid.value, color_by=w_color.value,
                     mask_alpha=w_alpha.value, contour_width=w_width.value,
                     filter_labels=list(w_filter.value) or None,
                     original_side=w_side.value, summary=w_sum.value)
            except Exception as exc:
                print(f"{type(exc).__name__}: {exc}")

    w_ds.observe(sync_split, "value")
    w_split.observe(sync_source, "value")
    w_src.observe(sync_images, "value")
    for w in (w_img, w_box, w_mask, w_cont, w_lab, w_iid, w_color,
              w_alpha, w_width, w_side, w_filter, w_sum):
        w.observe(refresh, "value")
    b_prev.on_click(step(-1))
    b_next.on_click(step(+1))

    ui = W.VBox([
        W.HBox([w_ds, w_split, w_src, w_img, b_prev, b_next]),
        W.HBox([w_box, w_mask, w_cont, w_lab, w_iid, w_sum]),
        W.HBox([w_color, w_alpha, w_width]),
        W.HBox([w_side, w_filter]),
    ])
    display(ui, out)
    sync_split()
    return {"ui": ui, "output": out, "refresh": refresh, "dataset": w_ds,
            "split": w_split, "source": w_src, "image": w_img, "classes": w_filter}


# `panel` keeps the widgets reachable from code: panel["image"].value = "000007.png"
panel = build_panel() if _HAS_WIDGETS else None

## Extras

- `grid(n)` — several images at once, overlay only.
- `compare_sources(image)` — the same image through every label source the dataset has.
- `save_overlay(path)` — write the overlay to PNG at the original resolution, no axes.
- `summarize_split(...)` — `label -> count` over a whole split.

In [ ]:
def _resolve(dataset=None, split=None, source=None, datasets_root=None):
    root = find_datasets_root(datasets_root if datasets_root is not None else DATASETS_ROOT)
    ds = resolve_dataset(root, dataset if dataset is not None else DATASET)
    split = split or SPLIT
    source = source or SOURCE
    return ds, split, source


def grid(n=6, start=0, ncols=3, dataset=None, split=None, source=None,
         filter_labels=None, **opts):
    """Mosaic of overlays (without the original next to each one)."""
    ds, split, source = _resolve(dataset, split, source)
    names = list_images(ds, split, source)[start:start + n]
    nrows = int(np.ceil(len(names) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.8 * nrows),
                             constrained_layout=True, squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, name in zip(axes.ravel(), names):
        img, records, id_map = build_scene(
            ds, split, name, source,
            FILTER_LABELS if filter_labels is None else filter_labels)
        colors = record_colors(records, opts.get("color_by", COLOR_BY))
        ax.imshow(overlay_image(img, records, id_map, colors,
                                show_mask=opts.get("show_mask", SHOW_MASK),
                                show_contour=opts.get("show_contour", SHOW_CONTOUR),
                                mask_alpha=opts.get("mask_alpha", MASK_ALPHA),
                                contour_width=opts.get("contour_width", CONTOUR_WIDTH)),
                  interpolation="nearest")
        if opts.get("show_box", SHOW_BOX) or opts.get("show_label", False):
            draw_annotations(ax, records, colors,
                             show_box=opts.get("show_box", SHOW_BOX),
                             show_label=opts.get("show_label", False),
                             show_inst_id=opts.get("show_inst_id", SHOW_INST_ID),
                             max_labels=opts.get("max_labels", MAX_LABELS),
                             max_boxes=opts.get("max_boxes", MAX_BOXES))
        ax.set_title(f"{name} — {len(records)} obj", fontsize=9)
        ax.set_xlim(-0.5, img.shape[1] - 0.5)
        ax.set_ylim(img.shape[0] - 0.5, -0.5)
    fig.suptitle(f"{ds.name} · {split} · {source}", fontsize=11)
    plt.show()


def save_overlay(path, dataset=None, split=None, source=None, image=None,
                 filter_labels=None, **opts):
    """Save the overlay alone, pixel for pixel, with no axes or margins."""
    ds, split, source = _resolve(dataset, split, source)
    names = list_images(ds, split, source)
    image = IMAGE if image is None else image
    name = names[int(image) % len(names)] if isinstance(image, (int, np.integer)) else str(image)
    img, records, id_map = build_scene(
        ds, split, name, source,
        FILTER_LABELS if filter_labels is None else filter_labels)
    colors = record_colors(records, opts.get("color_by", COLOR_BY))
    over = overlay_image(img, records, id_map, colors,
                         show_mask=opts.get("show_mask", SHOW_MASK),
                         show_contour=opts.get("show_contour", SHOW_CONTOUR),
                         mask_alpha=opts.get("mask_alpha", MASK_ALPHA),
                         contour_width=opts.get("contour_width", CONTOUR_WIDTH))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((over * 255).astype(np.uint8)).save(path)
    print(f"saved: {path}  ({len(records)} objects)")
    return path


# grid(6)
# save_overlay("overlay.png", source="instances")

In [ ]:
def compare_sources(image=None, dataset=None, split=None, sources=None,
                    filter_labels=None, **opts):
    """The same image through every available label source, plus the original."""
    ds, split, _ = _resolve(dataset, split, None)
    srcs = sources or available_sources(ds, split)
    names = list_images(ds, split, srcs[0])
    image = IMAGE if image is None else image
    name = names[int(image) % len(names)] if isinstance(image, (int, np.integer)) else str(image)

    panels = len(srcs) + 1
    ncols = min(panels, 3)
    nrows = int(np.ceil(panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 5.0 * nrows),
                             constrained_layout=True, squeeze=False)
    axs = axes.ravel()
    for ax in axs:
        ax.axis("off")

    base = load_image(ds, split, name, srcs[0])
    axs[0].imshow(as_rgb(base), interpolation="nearest")
    axs[0].set_title("original", fontsize=10)

    for ax, src in zip(axs[1:], srcs):
        try:
            img, records, id_map = build_scene(
                ds, split, name, src,
                FILTER_LABELS if filter_labels is None else filter_labels)
        except Exception as exc:
            ax.set_title(f"{src}: {type(exc).__name__}", fontsize=9)
            continue
        colors = record_colors(records, opts.get("color_by", COLOR_BY))
        ax.imshow(overlay_image(img, records, id_map, colors,
                                show_mask=opts.get("show_mask", SHOW_MASK),
                                show_contour=opts.get("show_contour", SHOW_CONTOUR),
                                mask_alpha=opts.get("mask_alpha", MASK_ALPHA),
                                contour_width=opts.get("contour_width", CONTOUR_WIDTH)),
                  interpolation="nearest")
        if opts.get("show_box", SHOW_BOX):
            draw_annotations(ax, records, colors, show_box=True, show_label=False,
                             max_boxes=opts.get("max_boxes", MAX_BOXES))
        ax.set_title(f"{src} — {len(records)} obj", fontsize=10)
        ax.set_xlim(-0.5, img.shape[1] - 0.5)
        ax.set_ylim(img.shape[0] - 0.5, -0.5)
    fig.suptitle(f"{ds.name} · {split} · {name}", fontsize=11)
    plt.show()


# compare_sources(0)

In [ ]:
# label -> count over a whole split.
# With `coco` it is instant; with `instances`/`unet` every PNG has to be opened,
# so use `limit=` to try it out first.

_ds, _split, _source = _resolve()
# _ = summarize_split(_ds, _split, _source, limit=50)